#1) Setup

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F
import requests

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

INPUT_PATH = "/Volumes/workspace/default/inputs"

arquivos_tabelas = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

hoje = datetime.now()
dbutils.widgets.text(
    "data_inicio",
    (hoje - timedelta(days=7)).strftime("%m-%d-%Y"),
    "Data inicial (MM-DD-AAAA)"
)
dbutils.widgets.text(
    "data_fim",
    hoje.strftime("%m-%d-%Y"),
    "Data final (MM-DD-AAAA)"
)

#2) Ingestão dos CSVs na Bronze

A Bronze preserva os dados de origem e adiciona apenas o timestamp de ingestão.

In [0]:
for arquivo, tabela in arquivos_tabelas.items():
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .csv(f"{INPUT_PATH}/{arquivo}")
        .withColumn("ingestion_datetime", F.current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"workspace.bronze.{tabela}")
    )

print("✓ CSVs carregados na Bronze")

✓ CSVs carregados na Bronze


#3) Cotação do dólar - Banco Central

In [0]:
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{data_inicio}'",
    "@dataFinalCotacao": f"'{data_fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

dados = response.json()["value"]

if not dados:
    raise ValueError("A API não retornou cotações para o período informado.")

df_cotacao = (
    spark.createDataFrame(dados)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

(
    df_cotacao.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.tb_cotacao_dolar")
)

print("✓ Cotação do dólar carregada na Bronze")

✓ Cotação do dólar carregada na Bronze
